# CIS 536/736: Computer Graphics / Game Engines
## Lab 3b: Export Pipeline (Geometry Nodes → OpenUSD → Unity 6)

**Module 6: OpenUSD Interchange & Rigging Fundamentals** 
**Target Environment:** Blender 4.2 LTS / Unity 6 LTS / Python 3.10+ (pxr) 
**Execution Time:** 120 Minutes

---
### 1. Lab Objective
Transitioning from procedural generation to game engine ingestion. You will take the parameterized asset from Lab 3a, generate multiple variations by manipulating the seed, and package them as an OpenUSD Variant Set. Finally, you will import this unified asset into Unity 6 and verify that the geometric variants swap cleanly at runtime.

### 2. Generating the Variants (Blender)
1. Open your completed `.blend` file from Lab 3a.
2. Locate the exposed `Seed` parameter in your Geometry Nodes modifier panel.
3. Set `Seed = 1`. Navigate to **File > Export > Universal Scene Description (.usd)**. Export as `asset_variant_A.usda`.
4. Set `Seed = 2`. Export as `asset_variant_B.usda`.
5. Set `Seed = 3`. Export as `asset_variant_C.usda`.

*(Crucial: Ensure 'Realize Instances' is active in your node tree, or the USD files will be empty).* 

### 3. Composing the Variant Set (Python `pxr` API)
You will use the Python script block below to computationally compose `A`, `B`, and `C` into a single master USD file containing a Variant Set. Do not attempt to merge these by hand in Blender.

In [ ]:
from pxr import Usd, UsdGeom, Sdf
import os

# Ensure your exported files are in the expected directory
export_dir = "exports/"
master_usd_path = os.path.join(export_dir, "master_asset_with_variants.usda")

def compose_variant_set():
    print("\n--- Composing USD Variant Set ---")
    
    # Create a new empty stage
    stage = Usd.Stage.CreateNew(master_usd_path)
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
    
    # Create the root Prim and define the VariantSet
    root_prim = stage.DefinePrim("/ProceduralAsset", "Xform")
    stage.SetDefaultPrim(root_prim)
    vset = root_prim.GetVariantSets().AddVariantSet("shapeVariant")
    
    # Append the variant payloads
    variants = {'VarA': 'asset_variant_A.usda', 'VarB': 'asset_variant_B.usda', 'VarC': 'asset_variant_C.usda'}
    
    for var_name, file_name in variants.items():
        if not os.path.exists(os.path.join(export_dir, file_name)):
             print(f"[ERROR] Missing variant file: {file_name}")
             return False
             
        vset.AddVariant(var_name)
        vset.SetVariantSelection(var_name)
        
        with vset.GetVariantEditContext():
            # Reference the external payload
            root_prim.GetReferences().AddReference(f"./{file_name}")
            print(f"[PASS] Added {var_name} referencing {file_name}")
            
    # Save the master file
    stage.GetRootLayer().Save()
    print(f"\n[SUCCESS] Master USD saved to: {master_usd_path}")
    return True

# Execute the composition
# compose_variant_set()

### 4. Engine Ingestion (Unity 6)
Now that the master `.usda` file is compiled, we ingest it into the runtime environment.

1. Open your Unity 6 HDRP project.
2. Ensure the **USD Package** is installed via the Package Manager.
3. Drag `master_asset_with_variants.usda` into your `Assets/Models/` folder.
4. Drag the asset from the Project window into your Scene Hierarchy.
5. Select the asset in the Hierarchy. In the Inspector, locate the USD component.
6. Find the `shapeVariant` dropdown. Cycle through `VarA`, `VarB`, and `VarC`. The geometry in the scene should swap instantaneously without loading a new prefab.

### 5. Fixing the Up-Axis & Scale (The Coordinate Trap)
Blender is Z-up. Unity is Y-up. If your asset imports lying on its side or appears microscopic:

* **Axis Fix:** Check the import settings on the USD asset in Unity. Ensure the `Scale` factor is set correctly (usually `100` if exported unscaled from Blender) and the Up-Axis conversion is checked.
* **Material Rebinding:** Assign the HDRP PBR material you authored in Lab 2a to the Mesh Renderer. The USD variant swap will retain the material assignment.

---
### 6. Submission Checklist & GitOps Protocol
- [ ] 3 distinct procedural variants exported to `.usda`.
- [ ] Python script successfully composed `master_asset_with_variants.usda`.
- [ ] Asset imported into Unity 6 and variants swap correctly via the Inspector.
- [ ] Orientation (Z-up to Y-up) and scale are correct in the Unity scene.
- [ ] The `.blend` source, the 4 `.usda` files, and a screenshot of the asset in Unity are committed to your `cis536_736_hw` repository.
- [ ] The absolute SSOT URL to your commit is posted in the Canvas assignment drop.